In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import (
    LightGBMModel, XGBModel, TiDEModel, TSMixerModel,
    DLinearModel, NHiTSModel, NLinearModel, RNNModel
)
from darts.metrics import mae, rmse, mape

# Import ISPU calculator
from ispu_calculator import calculate_ispu_for_dataframe, map_to_3_categories

print("✅ Imports loaded successfully")

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


✅ Imports loaded successfully


In [5]:
POLLUTANTS = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
WEATHER_FEATURES = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'precipitation_sum', 'wind_speed_10m_max', 'wind_speed_10m_mean',
    'relative_humidity_2m_mean', 'cloud_cover_mean', 'surface_pressure_mean'
]
STATIONS = ['DKI1', 'DKI2', 'DKI3', 'DKI4', 'DKI5']

df_train = pd.read_csv('../feature/final_feature.csv', parse_dates=['tanggal'])
df_forecast = pd.read_csv('../feature/forecast_features_sep_nov_2025.csv', parse_dates=['tanggal'])

df_train = df_train[df_train['stasiun'].isin(STATIONS)].copy()
df_train = df_train.dropna(subset=POLLUTANTS)
df_train.shape

NameError: name 'pd' is not defined

In [9]:
# Check GPU availability
import torch

print(f"{'='*60}")
print(f"Hardware Configuration")
print(f"{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    ACCELERATOR = 'gpu'
    DEVICES = 1
    BATCH_SIZE = 64  # Larger batch for GPU
    print(f"✅ Using GPU for training!")
else:
    ACCELERATOR = 'cpu'
    DEVICES = 'auto'
    BATCH_SIZE = 32  # Smaller batch for CPU
    print(f"⚠️ GPU not available, using CPU")

print(f"{'='*60}\n")

Hardware Configuration
PyTorch version: 2.10.0
CUDA available: True
CUDA version: 13.1
GPU device: NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory: 3.95 GB
✅ Using GPU for training!



In [10]:
def create_series_per_station(df, value_cols, station):
    df_st = df[df['stasiun'] == station].copy()
    df_st = df_st.sort_values('tanggal')
    
    # Handle duplicate dates by taking mean (aggregate multiple records per day)
    df_st = df_st.groupby('tanggal')[value_cols].mean()
    
    # Resample to daily frequency and fill missing values
    df_st = df_st.asfreq('D')
    df_st = df_st.ffill().bfill()
    return TimeSeries.from_dataframe(df_st, fill_missing_dates=True, freq='D')

target_series = {}
cov_series = {}
for st in STATIONS:
    target_series[st] = create_series_per_station(df_train, POLLUTANTS, st)
    cov_series[st] = create_series_per_station(df_train, WEATHER_FEATURES, st)

future_cov_series = {}
for st in STATIONS:
    future_cov_series[st] = create_series_per_station(df_forecast, WEATHER_FEATURES, st)

print(f"Target series length: {len(target_series['DKI1'])}")
print(f"Future cov series length: {len(future_cov_series['DKI1'])}")

Target series length: 5722
Future cov series length: 91


In [11]:
VAL_LENGTH = 91
train_targets, val_targets = {}, {}
train_covs, val_covs = {}, {}

for st in STATIONS:
    train_targets[st] = target_series[st][:-VAL_LENGTH]
    val_targets[st] = target_series[st][-VAL_LENGTH:]
    train_covs[st] = cov_series[st][:-VAL_LENGTH]
    val_covs[st] = cov_series[st][-VAL_LENGTH:]

train_cov_extended = {}
for st in STATIONS:
    train_cov_extended[st] = train_covs[st].append(val_covs[st])

print(f"Train: {len(train_targets['DKI1'])}, Val: {len(val_targets['DKI1'])}")

Train: 5631, Val: 91


In [12]:
INPUT_CHUNK = 21  # Reduced to work with shorter training periods
OUTPUT_CHUNK = 14
FORECAST_HORIZON = 91

# PyTorch Lightning trainer kwargs with GPU support
pl_kwargs = {
    'enable_progress_bar': True,
    'accelerator': ACCELERATOR,
    'devices': DEVICES,
    'enable_model_summary': True
}

models = {
    'LightGBM': LightGBMModel(
        lags=INPUT_CHUNK,
        lags_future_covariates=(INPUT_CHUNK, OUTPUT_CHUNK),
        output_chunk_length=OUTPUT_CHUNK,
        verbose=-1
    ),
    'XGBoost': XGBModel(
        lags=INPUT_CHUNK,
        lags_future_covariates=(INPUT_CHUNK, OUTPUT_CHUNK),
        output_chunk_length=OUTPUT_CHUNK,
        verbosity=0
    ),
    # 'TiDE': TiDEModel(
    #     input_chunk_length=INPUT_CHUNK,
    #     output_chunk_length=OUTPUT_CHUNK,
    #     use_reversible_instance_norm=True,
    #     n_epochs=100,  # Increased for GPU
    #     batch_size=BATCH_SIZE,
    #     pl_trainer_kwargs=pl_kwargs
    # ),
    # 'TSMixer': TSMixerModel(
    #     input_chunk_length=INPUT_CHUNK,
    #     output_chunk_length=OUTPUT_CHUNK,
    #     use_reversible_instance_norm=True,
    #     n_epochs=100,  # Increased for GPU
    #     batch_size=BATCH_SIZE,
    #     pl_trainer_kwargs=pl_kwargs
    # ),
    # 'DLinear': DLinearModel(
    #     input_chunk_length=INPUT_CHUNK,
    #     output_chunk_length=OUTPUT_CHUNK,
    #     n_epochs=100,  # Increased for GPU
    #     batch_size=BATCH_SIZE,
    #     pl_trainer_kwargs=pl_kwargs
    # ),
    # 'NHiTS': NHiTSModel(
    #     input_chunk_length=INPUT_CHUNK,
    #     output_chunk_length=OUTPUT_CHUNK,
    #     n_epochs=100,  # Increased for GPU
    #     batch_size=BATCH_SIZE,
    #     pl_trainer_kwargs=pl_kwargs
    # ),
    # 'NLinear': NLinearModel(
    #     input_chunk_length=INPUT_CHUNK,
    #     output_chunk_length=OUTPUT_CHUNK,
    #     n_epochs=100,  # Increased for GPU
    #     batch_size=BATCH_SIZE,
    #     pl_trainer_kwargs=pl_kwargs
    # ),
    'LSTM': RNNModel(
        model='LSTM',
        input_chunk_length=INPUT_CHUNK,
        output_chunk_length=OUTPUT_CHUNK,
        hidden_dim=64,
        n_rnn_layers=2,
        dropout=0.2,
        n_epochs=100,
        batch_size=BATCH_SIZE,
        pl_trainer_kwargs=pl_kwargs
    )
}


ignoring user defined `output_chunk_length`. RNNModel uses a fixed `output_chunk_length=1`.


In [13]:
def evaluate_model(model, train_list, val_list, cov_list):
    model.fit(series=train_list, future_covariates=cov_list)
    preds = model.predict(n=VAL_LENGTH, series=train_list, future_covariates=cov_list)
    
    mae_scores, rmse_scores = [], []
    for pred, val in zip(preds, val_list):
        mae_scores.append(mae(val, pred))
        rmse_scores.append(rmse(val, pred))
    
    return np.mean(mae_scores), np.mean(rmse_scores)

train_list = [train_targets[st] for st in STATIONS]
val_list = [val_targets[st] for st in STATIONS]
cov_list = [train_cov_extended[st] for st in STATIONS]

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    try:
        mae_score, rmse_score = evaluate_model(model, train_list, val_list, cov_list)
        results[name] = {'MAE': mae_score, 'RMSE': rmse_score}
        print(f"  MAE: {mae_score:.2f}, RMSE: {rmse_score:.2f}")
    except Exception as e:
        print(f"  Error: {e}")
        results[name] = {'MAE': np.nan, 'RMSE': np.nan}

Training LightGBM...


  MAE: 7.59, RMSE: 9.50
Training XGBoost...


KeyboardInterrupt: 

In [14]:
results_df = pd.DataFrame(results).T.sort_values('MAE')
results_df

,MAE,RMSE
LightGBM,7.58884,9.500209


In [15]:
best_model_name = results_df.index[0]
print(f"Best model: {best_model_name}")

best_model = models[best_model_name]

full_train_list = [target_series[st] for st in STATIONS]

full_cov_list = []
for st in STATIONS:
    combined = cov_series[st].append(future_cov_series[st])
    full_cov_list.append(combined)

best_model.fit(series=full_train_list, future_covariates=full_cov_list)
final_preds = best_model.predict(n=FORECAST_HORIZON, series=full_train_list, future_covariates=full_cov_list)

Best model: LightGBM


In [29]:
# Generate submission with proper ISPU calculation
print(f"{'='*60}")
print(f"Generating Submission with ISPU Calculator (Permen LHK 14/2020)")
print(f"{'='*60}\n")

submissions = []
pollutant_predictions = []  # Store raw pollutant predictions

for i, st in enumerate(STATIONS):
    # Get predictions - convert TimeSeries to DataFrame
    try:
        # Try the standard Darts method
        pred_df = final_preds[i].pd_dataframe()
    except AttributeError:
        # Fallback: use alternative methods
        try:
            pred_df = pd.DataFrame(
                final_preds[i].values(), 
                index=final_preds[i].time_index,
                columns=final_preds[i].components
            )
        except:
            # Last resort: use all_values()
            pred_df = pd.DataFrame(
                final_preds[i].all_values().squeeze(),
                index=final_preds[i].time_index,
                columns=POLLUTANTS
            )
    
    # Ensure non-negative values
    pred_df = pred_df.clip(lower=0)
    
    # Save raw pollutant predictions
    pollutant_pred_df = pred_df.copy()
    pollutant_pred_df['stasiun'] = st
    pollutant_pred_df['tanggal'] = pollutant_pred_df.index
    pollutant_predictions.append(pollutant_pred_df)
    
    # ✅ Use proper ISPU calculation with linear interpolation
    pred_df = calculate_ispu_for_dataframe(
        pred_df, 
        pollutant_cols=POLLUTANTS
    )
    
    # Map to 3 categories if needed (optional)
    pred_df['category_3class'] = pred_df['category'].apply(map_to_3_categories)
    
    # Create submission ID
    pred_df['id'] = pred_df.index.strftime('%Y-%m-%d') + '_' + st
    
    # Show sample for this station
    print(f"{st}: Categories distribution")
    print(pred_df['category'].value_counts())
    print(f"   Critical pollutants: {pred_df['critical_parameter'].value_counts().to_dict()}")
    print()
    
    submissions.append(pred_df[['id', 'category', 'category_3class', 'max_ispu', 'critical_parameter']])

submission_df = pd.concat(submissions, ignore_index=True)

# Save both 5-class and 3-class versions
submission_df[['id', 'category']].to_csv('submission_darts_5class.csv', index=False)
submission_df[['id', 'category_3class']].rename(columns={'category_3class': 'category'}).to_csv('submission_darts_3class.csv', index=False)

# Save raw pollutant predictions
pollutant_predictions_df = pd.concat(pollutant_predictions, ignore_index=True)
pollutant_predictions_df = pollutant_predictions_df[['tanggal', 'stasiun'] + POLLUTANTS]
pollutant_predictions_df.to_csv('predictions_pollutants.csv', index=False)

print(f"\n{'='*60}")
print(f"Submissions saved:")










submission_df.head(10)
print(f"\n📋 Submission preview:")
print(pollutant_predictions_df.head(10))
print(f"\n📊 Sample pollutant predictions:")
print(f"{'='*60}")
print(f"  - predictions_pollutants.csv (Raw pollutant concentrations)")
print(f"  - submission_darts_3class.csv (BAIK/SEDANG/TIDAK SEHAT)")
print(f"  - submission_darts_5class.csv (BAIK/SEDANG/TIDAK SEHAT/SANGAT TIDAK SEHAT/BERBAHAYA)")
print(f"\n{'='*60}")
print(f"Submissions saved:")
print(f"  - submission_darts_5class.csv (BAIK/SEDANG/TIDAK SEHAT/SANGAT TIDAK SEHAT/BERBAHAYA)")
print(f"  - submission_darts_3class.csv (BAIK/SEDANG/TIDAK SEHAT)")
print(f"{'='*60}")

submission_df.head(10)

Generating Submission with ISPU Calculator (Permen LHK 14/2020)

DKI1: Categories distribution
category
TIDAK SEHAT    89
SEDANG          2
Name: count, dtype: int64
   Critical pollutants: {'pm25': 91}

DKI2: Categories distribution
category
TIDAK SEHAT    91
Name: count, dtype: int64
   Critical pollutants: {'pm25': 91}

DKI3: Categories distribution
category
TIDAK SEHAT    89
SEDANG          2
Name: count, dtype: int64
   Critical pollutants: {'pm25': 91}

DKI4: Categories distribution
category
TIDAK SEHAT    86
SEDANG          5
Name: count, dtype: int64
   Critical pollutants: {'pm25': 91}

DKI5: Categories distribution
category
TIDAK SEHAT    90
SEDANG          1
Name: count, dtype: int64
   Critical pollutants: {'pm25': 91}


Submissions saved:

📋 Submission preview:
     tanggal stasiun       pm10       pm25        so2         co         o3  \
0 2025-09-01    DKI1  48.090330  78.855496  30.663011  13.014066  19.373216   
1 2025-09-02    DKI1  47.236086  81.925521  33.042287  12

,id,category,category_3class,max_ispu,critical_parameter
0,2025-09-01_DKI1,TIDAK SEHAT,TIDAK SEHAT,124.69,pm25
1,2025-09-02_DKI1,TIDAK SEHAT,TIDAK SEHAT,127.92,pm25
2,2025-09-03_DKI1,TIDAK SEHAT,TIDAK SEHAT,131.26,pm25
3,2025-09-04_DKI1,TIDAK SEHAT,TIDAK SEHAT,131.54,pm25
4,2025-09-05_DKI1,TIDAK SEHAT,TIDAK SEHAT,131.60,pm25
5,2025-09-06_DKI1,TIDAK SEHAT,TIDAK SEHAT,130.41,pm25
6,2025-09-07_DKI1,TIDAK SEHAT,TIDAK SEHAT,136.55,pm25
7,2025-09-08_DKI1,TIDAK SEHAT,TIDAK SEHAT,136.86,pm25
8,2025-09-09_DKI1,TIDAK SEHAT,TIDAK SEHAT,134.68,pm25
9,2025-09-10_DKI1,TIDAK SEHAT,TIDAK SEHAT,130.97,pm25


In [18]:
# Summary statistics
print(f"\n{'='*60}")
print(f"Final Submission Summary")

print(f"{'='*60}")
print(f"\n{'='*60}")

print(f"\nTotal predictions: {len(submission_df)}")
print(submission_df['max_ispu'].describe())
print(f"Expected: {len(STATIONS)} stations × {FORECAST_HORIZON} days = {len(STATIONS) * FORECAST_HORIZON}")
print(f"\nISPU Statistics:")

print(f"\n5-Class Category Distribution:")
print(submission_df['critical_parameter'].value_counts())

print(submission_df['category'].value_counts())
print(f"\nCritical Pollutants Overall:")  
print(f"\n3-Class Category Distribution:")
print(submission_df['category_3class'].value_counts())


Final Submission Summary


Total predictions: 455
count    455.000000
mean     122.153582
std        9.500610
min       95.160000
25%      115.795000
50%      123.290000
75%      129.320000
max      142.870000
Name: max_ispu, dtype: float64
Expected: 5 stations × 91 days = 455

ISPU Statistics:

5-Class Category Distribution:
critical_parameter
pm25    455
Name: count, dtype: int64
category
TIDAK SEHAT    445
SEDANG          10
Name: count, dtype: int64

Critical Pollutants Overall:

3-Class Category Distribution:
category_3class
TIDAK SEHAT    445
SEDANG          10
Name: count, dtype: int64


In [26]:
# Evaluation vs Ground Truth (if available)
print(f"\n{'='*60}")
print(f"Evaluation vs Ground Truth")
print(f"{'='*60}\n")

try:
    # Load ground truth
    gt = pd.read_csv('../dataset/data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data.csv')
    
    # Parse date from periode_data (YYYYMM format) + tanggal (day)
    # Extract year, month from periode_data and combine with day from tanggal
    gt['tanggal_parsed'] = pd.to_datetime(
        gt['periode_data'].astype(str) + gt['tanggal'].astype(str).str.zfill(2),
        format='%Y%m%d',
        errors='coerce'
    )
    gt['tanggal'] = gt['tanggal_parsed']
    gt.drop('tanggal_parsed', axis=1, inplace=True)
    
    # Map station names
    station_map = {
        'DKI1 Bunderan HI': 'DKI1', 
        'DKI1 Bundaran Hotel Indonesia (HI)': 'DKI1', 
        'DKI1 Bundaran Hotel Indonesia HI': 'DKI1',
        'DKI2 Kelapa Gading': 'DKI2',
        'DKI3 Jagakarsa': 'DKI3',
        'DKI4 Lubang Buaya': 'DKI4',
        'DKI5 Kebon Jeruk': 'DKI5',
        'DKI5 Kebon Jeruk Jakarta Barat': 'DKI5'
    }
    gt['stasiun'] = gt['stasiun'].map(station_map).fillna(gt['stasiun'])
    
    # Filter to forecast period (Sep-Nov 2025)
    gt_period = gt[(gt['tanggal'] >= '2025-09-01') & (gt['tanggal'] <= '2025-11-30')].copy()
    
    if len(gt_period) > 0:
        # Calculate ISPU for ground truth using proper formula
        gt_period = calculate_ispu_for_dataframe(gt_period, pollutant_cols=POLLUTANTS)
        gt_period['category_3class'] = gt_period['category'].apply(map_to_3_categories)
        gt_period['id'] = gt_period['tanggal'].dt.strftime('%Y-%m-%d') + '_' + gt_period['stasiun']
        
        # Merge predictions with ground truth
        merged = submission_df.merge(
            gt_period[['id', 'category', 'category_3class', 'max_ispu']], 
            on='id', 
            how='inner',
            suffixes=('_pred', '_true')
        )
        
        if len(merged) > 0:
            from sklearn.metrics import f1_score, classification_report, accuracy_score
            
            # Evaluate 5-class
            print("📊 5-Class Evaluation (BAIK/SEDANG/TIDAK SEHAT/SANGAT TIDAK SEHAT/BERBAHAYA):")
            print("-" * 60)
            labels_5 = ['BAIK', 'SEDANG', 'TIDAK SEHAT', 'SANGAT TIDAK SEHAT', 'BERBAHAYA']
            f1_5_macro = f1_score(merged['category_true'], merged['category_pred'], average='macro', labels=labels_5, zero_division=0)
            f1_5_weighted = f1_score(merged['category_true'], merged['category_pred'], average='weighted', labels=labels_5, zero_division=0)
            acc_5 = accuracy_score(merged['category_true'], merged['category_pred'])
            
            print(f"  Accuracy:        {acc_5:.4f}")
            print(f"  F1-Macro:        {f1_5_macro:.4f}")
            print(f"  F1-Weighted:     {f1_5_weighted:.4f}")
            print(f"  Matched samples: {len(merged)}\n")
            
            # Evaluate 3-class
            print("📊 3-Class Evaluation (BAIK/SEDANG/TIDAK SEHAT):")
            print("-" * 60)
            labels_3 = ['BAIK', 'SEDANG', 'TIDAK SEHAT']
            f1_3_macro = f1_score(merged['category_3class_true'], merged['category_3class_pred'], average='macro', labels=labels_3, zero_division=0)
            f1_3_weighted = f1_score(merged['category_3class_true'], merged['category_3class_pred'], average='weighted', labels=labels_3, zero_division=0)
            acc_3 = accuracy_score(merged['category_3class_true'], merged['category_3class_pred'])
            
            print(f"  Accuracy:        {acc_3:.4f}")
            print(f"  F1-Macro:        {f1_3_macro:.4f}")
            print(f"  F1-Weighted:     {f1_3_weighted:.4f}")
            print(f"\n{'='*60}")
            print("📋 Classification Report (3-Class):")
            print("="*60)
            print(classification_report(
                merged['category_3class_true'], 
                merged['category_3class_pred'],
                labels=labels_3,
                target_names=labels_3,
                zero_division=0
            ))
            
            # ISPU value correlation
            ispu_mae = np.mean(np.abs(merged['max_ispu_pred'] - merged['max_ispu_true']))
            ispu_rmse = np.sqrt(np.mean((merged['max_ispu_pred'] - merged['max_ispu_true'])**2))
            print(f"\n📈 ISPU Value Metrics:")
            print(f"  MAE:  {ispu_mae:.2f}")
            print(f"  RMSE: {ispu_rmse:.2f}")
            
        else:
            print("⚠️ No matching predictions found with ground truth")
    else:
        print("⚠️ No ground truth data found for forecast period (Sep-Nov 2025)")
        
except FileNotFoundError:
    print("ℹ️ Ground truth file not found - skipping evaluation")
except Exception as e:
    print(f"⚠️ Error during evaluation: {e}")


Evaluation vs Ground Truth

⚠️ Error during evaluation: Length of values (3) does not match length of index (455)


In [ ]:
# Visualization: Predictions vs Actual (if ground truth available)
import matplotlib.pyplot as plt
import seaborn as sns

if 'merged' in locals() and len(merged) > 0:
    print(f"\n{'='*60}")
    print(f"Visualizations")
    print(f"{'='*60}\n")
    
    # 1. Confusion Matrix - 3 Class
    from sklearn.metrics import confusion_matrix
    
    cm_3 = confusion_matrix(
        merged['category_3class_true'], 
        merged['category_3class_pred'],
        labels=['BAIK', 'SEDANG', 'TIDAK SEHAT']
    )
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_3, annot=True, fmt='d', cmap='Blues',
                xticklabels=['BAIK', 'SEDANG', 'TIDAK SEHAT'],
                yticklabels=['BAIK', 'SEDANG', 'TIDAK SEHAT'])
    plt.title(f'Confusion Matrix (3-Class)\nF1-Macro: {f1_3_macro:.4f}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    
    # 2. ISPU Value Scatter Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(merged['max_ispu_true'], merged['max_ispu_pred'], alpha=0.5, s=20)
    
    # Add perfect prediction line
    min_val = min(merged['max_ispu_true'].min(), merged['max_ispu_pred'].min())
    max_val = max(merged['max_ispu_true'].max(), merged['max_ispu_pred'].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')
    
    plt.xlabel('Actual ISPU')
    plt.ylabel('Predicted ISPU')
    plt.title(f'ISPU Predictions vs Actual\nMAE: {ispu_mae:.2f}, RMSE: {ispu_rmse:.2f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 3. Category distribution comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Actual
    merged['category_3class_true'].value_counts().reindex(['BAIK', 'SEDANG', 'TIDAK SEHAT'], fill_value=0).plot(
        kind='bar', ax=axes[0], color='steelblue'
    )
    axes[0].set_title('Actual Category Distribution')
    axes[0].set_ylabel('Count')
    axes[0].set_xlabel('Category')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Predicted
    merged['category_3class_pred'].value_counts().reindex(['BAIK', 'SEDANG', 'TIDAK SEHAT'], fill_value=0).plot(
        kind='bar', ax=axes[1], color='coral'
    )
    axes[1].set_title('Predicted Category Distribution')
    axes[1].set_ylabel('Count')
    axes[1].set_xlabel('Category')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # 4. Per-station performance
    print("\n📍 Per-Station Performance (3-Class F1-Macro):")
    print("-" * 60)
    for station in STATIONS:
        station_data = merged[merged['id'].str.endswith(station)]
        if len(station_data) > 0:
            station_f1 = f1_score(
                station_data['category_3class_true'],
                station_data['category_3class_pred'],
                average='macro',
                labels=['BAIK', 'SEDANG', 'TIDAK SEHAT'],
                zero_division=0
            )
            print(f"  {station}: {station_f1:.4f} ({len(station_data)} samples)")
    
else:
    print("ℹ️ No ground truth available for visualization")

ℹ️ No ground truth available for visualization


In [34]:
import importlib
import ispu_calculator

# 1. Reload modul agar perubahan kode terbaca
importlib.reload(ispu_calculator)
from ispu_calculator import calculate_ispu_for_dataframe, map_to_3_categories

print("✅ Modul ISPU berhasil direload dengan aturan baru (Looser Breakpoints).")

# 2. Jalankan ulang pembuatan submission
# (Pastikan 'final_preds' sudah ada atau sudah ditraining sebelumnya)
# ... COPY PASTE KODE SUBMISSION DI SINI ...

✅ Modul ISPU berhasil direload dengan aturan baru (Looser Breakpoints).


In [35]:
# Generate submission with proper ISPU calculation
print(f"{'='*60}")
print(f"Generating Submission with ISPU Calculator (Dataset-Compatible)")
print(f"{'='*60}\n")

submissions = []
pollutant_predictions = []  # Store raw pollutant predictions

for i, st in enumerate(STATIONS):
    # Get predictions - convert TimeSeries to DataFrame (Robust method)
    # Note: .pd_dataframe() might fail on some Darts versions, manual construction is safer
    pred_df = pd.DataFrame(
        final_preds[i].values(), 
        index=final_preds[i].time_index,
        columns=final_preds[i].components
    )
    
    # Ensure non-negative values
    pred_df = pred_df.clip(lower=0)
    
    # Save raw pollutant predictions
    pollutant_pred_df = pred_df.copy()
    pollutant_pred_df['stasiun'] = st
    pollutant_pred_df['tanggal'] = pollutant_pred_df.index
    pollutant_predictions.append(pollutant_pred_df)
    
    # ✅ Use proper ISPU calculation (Uses updated ispu_calculator.py logic)
    pred_df = calculate_ispu_for_dataframe(
        pred_df, 
        pollutant_cols=POLLUTANTS
    )
    
    # Map to 3 categories if needed (optional)
    pred_df['category_3class'] = pred_df['category'].apply(map_to_3_categories)
    
    # Create submission ID
    pred_df['id'] = pred_df.index.strftime('%Y-%m-%d') + '_' + st
    
    # Show sample for this station
    print(f"{st}: Categories distribution")
    print(pred_df['category'].value_counts())
    print(f"   Critical pollutants: {pred_df['critical_parameter'].value_counts().to_dict()}")
    print()
    
    submissions.append(pred_df[['id', 'category', 'category_3class', 'max_ispu', 'critical_parameter']])

submission_df = pd.concat(submissions, ignore_index=True)

# Save both 5-class and 3-class versions
submission_df[['id', 'category']].to_csv('submission_darts_5class.csv', index=False)
submission_df[['id', 'category_3class']].rename(columns={'category_3class': 'category'}).to_csv('submission_darts_3class.csv', index=False)

# Save raw pollutant predictions
pollutant_predictions_df = pd.concat(pollutant_predictions, ignore_index=True)
pollutant_predictions_df = pollutant_predictions_df[['tanggal', 'stasiun'] + POLLUTANTS]
pollutant_predictions_df.to_csv('predictions_pollutants.csv', index=False)

print(f"\n{'='*60}")
print(f"Submissions saved:")
print(f"  - predictions_pollutants.csv (Raw pollutant concentrations)")
print(f"  - submission_darts_3class.csv (BAIK/SEDANG/TIDAK SEHAT)")
print(f"  - submission_darts_5class.csv (BAIK/SEDANG/TIDAK SEHAT/SANGAT TIDAK SEHAT/BERBAHAYA)")
print(f"{'='*60}")

submission_df.head(10)

Generating Submission with ISPU Calculator (Dataset-Compatible)

DKI1: Categories distribution
category
SEDANG         73
BAIK           14
TIDAK SEHAT     4
Name: count, dtype: int64
   Critical pollutants: {'pm25': 90, 'pm10': 1}

DKI2: Categories distribution
category
SEDANG    77
BAIK      14
Name: count, dtype: int64
   Critical pollutants: {'pm25': 91}

DKI3: Categories distribution
category
SEDANG    71
BAIK      20
Name: count, dtype: int64
   Critical pollutants: {'pm25': 85, 'pm10': 6}

DKI4: Categories distribution
category
SEDANG         74
BAIK           16
TIDAK SEHAT     1
Name: count, dtype: int64
   Critical pollutants: {'pm25': 90, 'pm10': 1}

DKI5: Categories distribution
category
SEDANG         76
BAIK           11
TIDAK SEHAT     4
Name: count, dtype: int64
   Critical pollutants: {'pm25': 90, 'pm10': 1}


Submissions saved:
  - predictions_pollutants.csv (Raw pollutant concentrations)
  - submission_darts_3class.csv (BAIK/SEDANG/TIDAK SEHAT)
  - submission_darts_5

,id,category,category_3class,max_ispu,critical_parameter
0,2025-09-01_DKI1,SEDANG,SEDANG,77.71,pm25
1,2025-09-02_DKI1,SEDANG,SEDANG,83.85,pm25
2,2025-09-03_DKI1,SEDANG,SEDANG,90.20,pm25
3,2025-09-04_DKI1,SEDANG,SEDANG,90.73,pm25
4,2025-09-05_DKI1,SEDANG,SEDANG,90.83,pm25
5,2025-09-06_DKI1,SEDANG,SEDANG,88.58,pm25
6,2025-09-07_DKI1,BAIK,BAIK,100.12,pm25
7,2025-09-08_DKI1,BAIK,BAIK,100.38,pm25
8,2025-09-09_DKI1,SEDANG,SEDANG,96.69,pm25
9,2025-09-10_DKI1,SEDANG,SEDANG,89.65,pm25
